# SpectraLite — Research Lab (`works.ipynb`)

**Single notebook for the whole project.** Open this file via:

`File → Open notebook → GitHub → SpectraLite/notebooks/works.ipynb`

Do **not** double-click notebooks in the left Files panel (raw text mode).

| Stage | Status in this file | Depends on |
|-------|---------------------|------------|
| 0 Bootstrap + env | **Runnable now** | Fresh Colab GPU runtime |
| Phase 0 smoke test | **Runnable now** | Stage 0 |
| Phase 1 baseline harness | Placeholder | Phase 0 PASS on GPU |
| Phase 2 vanilla SVD | Placeholder | Phase 1 metrics schema |
| Phase 3 ASVD / SVD-LLM | Placeholder | Phase 2 |
| Phase 4 SpectraLite ranks | Placeholder | Phase 3 |
| Phase 5 stability | Placeholder | Phase 4 |
| Phase 6 latency engineering | Placeholder | Phase 5 |
| Phase 7 ablations | Placeholder | Phase 6 |
| Phase 8 downstream eval | Placeholder | Chosen operating point |

When a stage is ready, we fill its section here. If it needs prior outputs, run the previous section, save/push this notebook, and tell Cursor to continue.


---
# Stage 0 — Colab Bootstrap

**Run once per new Colab runtime** (A100 recommended).

- Clones / pulls the full GitHub repo
- Installs `requirements-colab.txt` (does **not** reinstall torch)
- Verifies CUDA


### 0.1 Runtime check

In [ ]:
import sys
print("In Colab:", "google.colab" in sys.modules)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        raise SystemExit(
            "Select Runtime → Change runtime type → GPU (A100), "
            "then Disconnect and delete runtime if CUDA stays False."
        )
except ImportError as exc:
    raise SystemExit(f"torch missing: {exc}") from exc


### 0.2 Clone or pull repo

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/PrabinDevkota/Execution.git"
CLONE_ROOT = Path("/content/Execution")
PACKAGE_ROOT = CLONE_ROOT / "SpectraLite"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not PACKAGE_ROOT.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(CLONE_ROOT)])
    else:
        subprocess.check_call(["git", "-C", str(CLONE_ROOT), "pull", "--ff-only"])
    if str(PACKAGE_ROOT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_ROOT))
    %cd {PACKAGE_ROOT}
else:
    # Local: expect cwd inside SpectraLite or notebooks/
    cwd = Path.cwd().resolve()
    PACKAGE_ROOT = None
    for candidate in [cwd, cwd.parent]:
        if (candidate / "spectralite").is_dir():
            PACKAGE_ROOT = candidate
            break
    if PACKAGE_ROOT is None:
        raise FileNotFoundError("Run from SpectraLite repo locally.")
    if str(PACKAGE_ROOT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_ROOT))

print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("spectralite OK =", (PACKAGE_ROOT / "spectralite").is_dir())


### 0.3 Install dependencies + CUDA verify

In [ ]:
from spectralite.colab_setup import ensure_environment, in_colab

# On Colab: install requirements-colab.txt (no torch). Local: requirements.txt.
repo_root = ensure_environment(
    pull=False,
    install=True,
    require_cuda_on_colab=True,
)
print("Stage 0 complete. repo_root =", repo_root)


---
# Phase 0 — Environment & Model Smoke Test

**Goal:** Load `facebook/opt-125m` in FP16 on CUDA, inventory every `nn.Linear`, run one generation, report GPU memory.

**No compression / SVD / whitening here.**


### P0.1 Config, seed, env report

In [ ]:
from spectralite import __version__, default_config, set_seed, gpu
from spectralite.system import print_environment_report
from spectralite.utils import get_logger, print_section

cfg = default_config()
cfg.ensure_directories()
set_seed(cfg.seed)
logger = get_logger("spectralite.works")

print_section("SpectraLite works.ipynb — Phase 0")
print(f"  package     : {__version__}")
print(f"  model       : {cfg.model_name}")
print(f"  seed        : {cfg.seed}")
print(f"  dtype       : {cfg.dtype}")
print(f"  device_map  : {cfg.device_map}")

env_info = print_environment_report()
environment_ready = env_info["pytorch_version"] != "unavailable"
gpu_ready = bool(env_info["cuda_available"])
assert gpu_ready, "GPU required for SpectraLite Phase 0+"


### P0.2 GPU memory before load

In [ ]:
mem_before = gpu.snapshot(label="Memory before loading")
gpu.print_memory_snapshot(mem_before)


### P0.3 Load model + tokenizer

In [ ]:
from spectralite.model_loader import load_model_and_tokenizer, print_model_load_summary

model, tokenizer = load_model_and_tokenizer(config=cfg)
load_summary = print_model_load_summary(model, model_name=cfg.model_name)

mem_after_load = gpu.snapshot(label="Memory after loading")
gpu.print_memory_snapshot(mem_after_load)

model_loaded = model is not None and tokenizer is not None
logger.info("Loaded %s on %s", cfg.model_name, load_summary["device"])


### P0.4 Model analysis — every `nn.Linear`

In [ ]:
from spectralite.model_analysis import print_full_model_analysis

analysis = print_full_model_analysis(model, model_name=cfg.model_name)
print(f"\nBlocks (first 3): {analysis['block_names'][:3]}")
print(f"Linear layers: {analysis['num_linear_layers']}")

# Keep for later phases (SVD targets)
LINEAR_LAYERS = analysis["linear_layers"]


### P0.5 Test inference

In [ ]:
from spectralite.model_loader import generate_text

generated = generate_text(
    model,
    tokenizer,
    cfg.smoke_prompt,
    max_new_tokens=cfg.max_new_tokens,
    do_sample=False,
)

print_section("Test Inference")
print("Prompt:", cfg.smoke_prompt)
print("-" * 60)
print(generated)
print("-" * 60)

inference_ok = isinstance(generated, str) and len(generated.strip()) > 0


### P0.6 GPU memory after inference + checklist

In [ ]:
from spectralite.utils import print_checklist

mem_after_infer = gpu.snapshot(label="Memory after inference")
gpu.print_memory_snapshot(mem_after_infer)
gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])

phase0_complete = all([environment_ready, gpu_ready, model_loaded, inference_ok])
print_checklist(
    [
        ("Environment Ready", environment_ready),
        ("GPU Ready", gpu_ready),
        ("Model Loaded Successfully", model_loaded),
        ("Inference Successful", inference_ok),
        ("Phase 0 Complete", phase0_complete),
    ]
)

print_section("Phase 0 Outcome")
for line in [
    "Environment Ready",
    "GPU Ready",
    "Model Loaded Successfully",
    "Inference Successful",
    "Phase 0 Complete",
]:
    print(" ", line)

assert phase0_complete, "Phase 0 incomplete — fix GPU/load/inference before Phase 1"


---
# Phase 1 — Baseline Profiling Harness

**Status:** Not implemented yet (code will be added here).

**Depends on:** Phase 0 PASS (GPU + model load path verified).

**Will add:**
- FLOP counting (`FlopCounterMode`, `calflops`)
- Prefill / decode latency (`torch.cuda.Event`)
- WikiText-2 / PTB / C4 perplexity
- `results.csv` schema
- Dense baselines for OPT-125M (then ladder models)

**What you should do when ready:** Run Phase 0 above, **Save a copy in GitHub**, tell Cursor: *“Phase 0 done in works.ipynb — implement Phase 1.”*


In [ ]:
# PHASE 1 PLACEHOLDER — do not expect this to run yet
raise NotImplementedError(
    "Phase 1 baseline harness is not in works.ipynb yet. "
    "Finish Phase 0, push notebook outputs, then ask Cursor to implement Phase 1 here."
)


---
# Phase 2 — Vanilla Truncated SVD

**Status:** Placeholder.

**Depends on:** Phase 1 harness + `LINEAR_LAYERS` inventory from Phase 0.

**Will add:** Eckart–Young truncation, `LowRankLinear`, uniform rank ratios, PPL/FLOP/latency curves.


In [ ]:
# PHASE 2 PLACEHOLDER
raise NotImplementedError("Phase 2 vanilla SVD not implemented yet.")


---
# Phase 3 — Activation-Aware Baselines (ASVD / SVD-LLM)

**Status:** Placeholder.

**Depends on:** Phase 2 comparison tables + calibration data pipeline from Phase 1.


In [ ]:
# PHASE 3 PLACEHOLDER
raise NotImplementedError("Phase 3 activation-aware baselines not implemented yet.")


---
# Phase 4 — SpectraLite Spectral Rank Allocation (novelty core)

**Status:** Placeholder.

**Depends on:** Phase 3 whitening utilities + Phase 1 FLOP budget accounting.


In [ ]:
# PHASE 4 PLACEHOLDER
raise NotImplementedError("Phase 4 SpectraLite rank allocation not implemented yet.")


---
# Phase 5 — Stability Safeguards

**Status:** Placeholder.

**Depends on:** Phase 4 allocator.

**Will add:** Ledoit–Wolf shrinkage, condition-number gating, few-sample robustness.


In [ ]:
# PHASE 5 PLACEHOLDER
raise NotImplementedError("Phase 5 stability safeguards not implemented yet.")


---
# Phase 6 — Latency Gate + Real Speedup Engineering

**Status:** Placeholder.

**Depends on:** Phase 5 compressed checkpoints + Phase 1 latency harness.

**Will add:** `r < κ_speed · mn/(m+n)` gate, factor fusion, packed MLP, CUDA-graph decode.


In [ ]:
# PHASE 6 PLACEHOLDER
raise NotImplementedError("Phase 6 latency engineering not implemented yet.")


---
# Phase 7 — Ablations

**Status:** Placeholder.

**Depends on:** Phases 4–6 complete enough to toggle components.


In [ ]:
# PHASE 7 PLACEHOLDER
raise NotImplementedError("Phase 7 ablations not implemented yet.")


---
# Phase 8 — Downstream Evaluation

**Status:** Placeholder.

**Depends on:** Chosen compression operating points from Phases 4–6.

**Will add:** lm-eval zero-shot suite; optional encoder GLUE bridge.


In [ ]:
# PHASE 8 PLACEHOLDER
raise NotImplementedError("Phase 8 downstream eval not implemented yet.")


---
# How to continue with Cursor

1. Open **this** notebook from GitHub (not Files double-click).
2. Run **Stage 0** then **Phase 0** only.
3. `File → Save a copy in GitHub`.
4. Message Cursor: which phase to implement next (e.g. Phase 1).
5. `git pull` in Stage 0.2 and run the new section.

Older notebooks (`Colab_Bootstrap.ipynb`, `Phase0_Setup.ipynb`) are legacy; **`works.ipynb` is the main lab file.**
